# DrenaVision - treinamento reproduzível no Colab

Este notebook chama os mesmos scripts do repositório para pesquisar fontes, baixar e converter dados, validar splits, treinar, avaliar e executar inferência. Ele não contém métricas ou outputs pré-preenchidos. Se uma célula falhar, corrija a causa em vez de substituir o resultado por valores manuais.

Ative uma GPU em **Ambiente de execução > Alterar tipo de ambiente de execução** antes do treino.

## 1. Obter o repositório

Preencha a URL pública ou uma URL para a qual sua sessão tenha autorização. Não coloque token diretamente no notebook.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = ""  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
PROJECT = Path("/content/drenavision")

if PROJECT.joinpath("pyproject.toml").is_file():
    print(f"Usando checkout existente: {PROJECT}")
else:
    if not REPO_URL.strip():
        raise ValueError("Preencha REPO_URL com o repositório autorizado.")
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, str(PROJECT)],
        check=True,
    )
os.chdir(PROJECT)
print(subprocess.run(["git", "rev-parse", "HEAD"], check=True, text=True, capture_output=True).stdout.strip())


## 2. Instalar o ambiente e verificar a GPU

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-dev.txt"], check=True)


In [ ]:
import torch
print({
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_runtime": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
})
if not torch.cuda.is_available():
    print("AVISO: treino em CPU pode ser muito lento. Confirme o runtime GPU.")


## 3. Persistência opcional no Google Drive

A sessão do Colab é efêmera. Monte o Drive para copiar artefatos ao final; o pipeline continua trabalhando em `/content` para melhor desempenho.

In [ ]:
USE_DRIVE = False  # @param {type:"boolean"}
DRIVE_OUTPUT = Path("/content/drive/MyDrive/drenavision-artifacts")
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)


## 4. Registrar pesquisa e disponibilidade das fontes

In [ ]:
Path("artifacts").mkdir(exist_ok=True)
subprocess.run([
    sys.executable, "scripts/research_datasets.py",
    "--check-access",
    "--output", "artifacts/dataset_research.json",
], cwd=PROJECT, check=True)


## 5. Baixar fontes aprovadas

O download ocupa centenas de megabytes. Arquivos são verificados por tamanho/checksum quando o catálogo oferece esses dados. CLICK usa HTTP Range para obter somente a amostra 100/20/20.

In [ ]:
subprocess.run([
    sys.executable, "scripts/download_datasets.py",
    "--source", "all",
    "--seed", "42",
    "--click-train", "100",
    "--click-val", "20",
    "--click-test", "20",
], cwd=PROJECT, check=True)


## 6. Converter anotações para o esquema YOLO DrenaVision

In [ ]:
subprocess.run([
    sys.executable, "scripts/convert_annotations.py",
    "--raw-root", "datasets/raw",
    "--output-root", "datasets/processed/drenavision",
    "--manifest", "datasets/manifests/dataset_manifest.csv",
    "--seed", "42",
], cwd=PROJECT, check=True)


## 7. Validar integridade, classes, caixas e vazamento

A opção `--phash` detecta imagens visualmente próximas em splits diferentes. O treino não deve continuar se o comando retornar erro.

In [ ]:
subprocess.run([
    sys.executable, "scripts/validate_dataset.py",
    "--dataset-root", "datasets/processed/drenavision",
    "--yaml", "datasets/processed/drenavision/dataset.yaml",
    "--manifest", "datasets/manifests/dataset_manifest.csv",
    "--phash",
    "--phash-threshold", "4",
    "--report", "artifacts/dataset_validation.json",
], cwd=PROJECT, check=True)


## 8. Inspecionar visualmente uma amostra de anotações

In [ ]:
subprocess.run([
    sys.executable, "scripts/visualize_annotations.py",
    "--dataset-root", "datasets/processed/drenavision",
    "--output-root", "artifacts/dataset_preview",
    "--samples-per-split", "6",
    "--seed", "42",
], cwd=PROJECT, check=True)

from IPython.display import display
from PIL import Image
preview_paths = sorted(Path("artifacts/dataset_preview").rglob("*.jpg"))[:6]
for preview in preview_paths:
    print(preview)
    display(Image.open(preview))


## 9. Executar fine-tuning customizado

O treino completo usa 100 épocas como máximo e early stopping. Para um teste técnico, reduza `EPOCHS`, mas não apresente o checkpoint de smoke test como resultado final.

In [ ]:
EPOCHS = 100  # @param {type:"integer"}
BATCH = 8  # @param {type:"integer"}
DEVICE = "0" if torch.cuda.is_available() else "cpu"
RUN_NAME = "yolo26n_drenavision_colab"  # @param {type:"string"}

subprocess.run([
    sys.executable, "scripts/train.py",
    "--config", "configs/train.yaml",
    "--epochs", str(EPOCHS),
    "--batch", str(BATCH),
    "--device", DEVICE,
    "--name", RUN_NAME,
], cwd=PROJECT, check=True)


## 10. Avaliar exclusivamente no split de teste

In [ ]:
EVAL_NAME = "test_yolo26n_drenavision_colab"  # @param {type:"string"}
subprocess.run([
    sys.executable, "scripts/evaluate.py",
    "--config", "configs/evaluate.yaml",
    "--model", "artifacts/models/best.pt",
    "--device", DEVICE,
    "--name", EVAL_NAME,
], cwd=PROJECT, check=True)


In [ ]:
import json
metric_files = sorted(Path("artifacts/evaluation").rglob("metrics.json"), key=lambda p: p.stat().st_mtime)
if not metric_files:
    raise FileNotFoundError("A avaliação não gerou metrics.json")
metrics_path = metric_files[-1]
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
print("Arquivo autoritativo:", metrics_path)
print(json.dumps(metrics["metrics"], indent=2, ensure_ascii=False))


## 11. Inferência real em uma imagem de teste

In [ ]:
test_images = sorted(Path("datasets/processed/drenavision/images/test").glob("*"))
test_images = [p for p in test_images if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
if not test_images:
    raise FileNotFoundError("O split test não contém imagens")
test_image = test_images[0]
subprocess.run([
    sys.executable, "scripts/predict.py",
    "--input", str(test_image),
    "--model", "artifacts/models/best.pt",
    "--output-dir", "artifacts/predictions",
    "--device", DEVICE,
], cwd=PROJECT, check=True)
annotated = Path("artifacts/predictions") / f"{test_image.stem}_annotated.jpg"
display(Image.open(annotated))


## 12. Preservar checkpoint, manifests e métricas

O pacote não inclui o dataset bruto. Verifique os hashes e copie o ZIP para armazenamento persistente antes de encerrar a sessão.

In [ ]:
import hashlib
import shutil

checkpoint = Path("artifacts/models/best.pt")
if not checkpoint.is_file():
    raise FileNotFoundError(checkpoint)
sha256 = hashlib.sha256(checkpoint.read_bytes()).hexdigest()
print("best.pt SHA-256:", sha256)

bundle_root = Path("/content/drenavision_evidence")
if bundle_root.exists():
    shutil.rmtree(bundle_root)
bundle_root.mkdir()
for source in [
    Path("artifacts/models"),
    Path("artifacts/training"),
    Path("artifacts/evaluation"),
    Path("artifacts/dataset_validation.json"),
    Path("artifacts/dataset_research.json"),
    Path("datasets/manifests"),
    Path("configs"),
]:
    if source.is_dir():
        shutil.copytree(source, bundle_root / source, dirs_exist_ok=True)
    elif source.is_file():
        target = bundle_root / source
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, target)
archive = Path(shutil.make_archive("/content/drenavision_evidence", "zip", bundle_root))
print("Pacote:", archive, archive.stat().st_size, "bytes")
if USE_DRIVE:
    shutil.copy2(archive, DRIVE_OUTPUT / archive.name)
    print("Cópia no Drive:", DRIVE_OUTPUT / archive.name)


In [ ]:
DOWNLOAD_NOW = False  # @param {type:"boolean"}
if DOWNLOAD_NOW:
    from google.colab import files
    files.download(str(archive))


## Checklist de encerramento

- confira `training_manifest.json` e `metrics.json`;
- registre o commit e o SHA-256 de `best.pt`;
- salve o ZIP fora da sessão;
- não copie métricas para o relatório sem vinculá-las ao checkpoint/dataset;
- não declare treino concluído se qualquer célula relevante falhou.